# Lesson 7: Multi-Step Execution

## Objective
Build agents that execute multi-step arithmetic problems requiring sequential tool calls. Track state across steps.

## Problem Statement
Previous lessons showed single or manual multi-step execution. Real problems are naturally sequential: "add these, then multiply, then divide." How can agents automatically chain operations?

## What's New
- **State Management**: Track variables and intermediate results
- **Sequential Chaining**: Operations depend on previous results
- **Variable Binding**: Store and reuse results
- **Looping**: Repeat until goal achieved

# Setup

Packages already installed: pip install -q python-dotenv google-cloud-aiplatform langchain-google-vertexai langgraph

In [ ]:
from dotenv import load_dotenv
import warnings
warnings.filterwarnings("ignore")
import os, vertexai
from langchain_google_vertexai import ChatVertexAI
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from typing import TypedDict, Dict
from langgraph.graph import StateGraph, START, END
import re
load_dotenv()
vertexai.init(
    project=os.getenv("PROJECT_ID"),
    location=os.getenv("LOCATION")
)
llm = ChatVertexAI(model="gemini-2.5-flash", temperature=0)
@tool
def add(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b
@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together."""
    return a * b
@tool
def divide(a: float, b: float) -> float:
    """Divide two numbers."""
    return a / b if b != 0 else 0
tools_dict = {"add": add, "multiply": multiply, "divide": divide}
print("✓ Setup complete")

# Define Multi-Step State

In [ ]:
class MultiStepState(TypedDict):
    problem: str
    steps_remaining: list  # Queue of operations to perform
    variables: Dict[str, float]  # Store intermediate results
    execution_log: list
    current_step: int
    max_steps: int
print("✓ Multi-step state defined")

# Implement Multi-Step Execution Nodes

In [ ]:
def planner(state: MultiStepState):
    """Break problem into steps."""
    problem = state["problem"]
    prompt = f"""Problem: {problem}
    Break this into arithmetic steps. Format each as:
    result_name = operation(arg1, arg2)
    Example for '(5+3)*2 = ?':
    temp1 = add(5, 3)
    final = multiply(temp1, 2)
    Just list the steps, one per line.
    """
    response = llm.invoke([HumanMessage(content=prompt)])
    lines = response.content.strip().split('\n')
    state["steps_remaining"] = [l.strip() for l in lines if l.strip()]
    state["execution_log"].append(f"Planned {len(state['steps_remaining'])} steps")
    return state
def executor(state: MultiStepState):
    """Execute next step from queue."""
    if not state["steps_remaining"] or state["current_step"] >= state["max_steps"]:
        state["execution_log"].append("Execution complete")
        return state
    step = state["steps_remaining"].pop(0)
    state["execution_log"].append(f"Step {state['current_step']}: {step}")
    # Parse: var_name = operation(arg1, arg2)
    match = re.match(r'(\w+)\s*=\s*(\w+)\((.*?)\)', step)
    if match:
        var_name = match.group(1)
        op_name = match.group(2)
        args_str = match.group(3)
        # Replace variable references with values
        for var, val in state["variables"].items():
            args_str = args_str.replace(var, str(val))
        try:
            args = [float(x.strip()) for x in args_str.split(',')]
            result = tools_dict[op_name].invoke({"a": args[0], "b": args[1]})
            state["variables"][var_name] = result
            state["execution_log"].append(f"  → {var_name} = {result}")
        except Exception as e:
            state["execution_log"].append(f"  → Error: {e}")
    state["current_step"] += 1
    return state
print("✓ Executor nodes implemented")

# Build Multi-Step Graph

In [ ]:
def should_continue_execution(state: MultiStepState):
    return "exec" if state["steps_remaining"] else "end"
graph_builder = StateGraph(MultiStepState)
graph_builder.add_node("plan", planner)
graph_builder.add_node("execute", executor)
graph_builder.add_edge(START, "plan")
graph_builder.add_edge("plan", "execute")
graph_builder.add_conditional_edges(
    "execute",
    should_continue_execution,
    {"exec": "execute", "end": END}
)
graph = graph_builder.compile()
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))


# Test Multi-Step Execution

Verify

In [ ]:
problem = "Calculate (10 + 5) * 2 / 3"
state = {
    "problem": problem,
    "steps_remaining": [],
    "variables": {},
    "execution_log": [],
    "current_step": 1,
    "max_steps": 10
}
print(f"Problem: {problem}\n")
result = graph.invoke(state)
print("Execution Log:")
for entry in result["execution_log"]:
    print(f"  {entry}")
print(f"\nVariables at end: {result['variables']}")
if "final" in result["variables"]:
    expected = (10 + 5) * 2 / 3
    actual = result["variables"]["final"]
    print(f"\nExpected: {expected}")
    print(f"Actual: {actual}")
elif "temp1" in result["variables"]:
    print(f"Partial result available")

# Key Insight: State as Memory

In [ ]:
print("""
Multi-Step State Management
============================
State Roles:
1. steps_remaining: QUEUE - what's left to do
2. variables: MEMORY - store results
3. execution_log: AUDIT TRAIL - what happened
4. current_step: COUNTER - prevent infinite loops
This pattern is used in:
- Database query execution (subquery results)
- Compiler intermediate representations
- Workflow engines (Airflow, Temporal)
- Robotic process automation
""")

# Verification

In [ ]:
print("="*50)
print("VERIFICATION - LESSON 7")
print("="*50)
print(f"✓ Multi-step state with queuing")
print(f"✓ Planner breaks problems into steps")
print(f"✓ Executor runs steps sequentially")
print(f"✓ Variable binding across steps")
print(f"✓ Loop control prevents infinite execution")
print(f"✓ Audit trail of all operations")
print(f"\n✓ LESSON 7 COMPLETE: Multi-step execution working!")
print("="*50)